# 03 — Heston–Merton

**What this notebook is:** a complete start-to-finish guide for Heston stochastic volatility **plus** Merton jumps, with an interactive Monte Carlo playground.

**Theory handbook:** [`00_MODELS_EXPLAINED.ipynb`](00_MODELS_EXPLAINED.ipynb) · **Related:** [`02_merton.ipynb`](02_merton.ipynb) (jumps only) · [`04_garch_merton.ipynb`](04_garch_merton.ipynb) (another vol model)

**Your project data:** `../data/equity/prices_clean.csv`, `log_returns_*.csv`, `summary_stats.csv`


## 1. Model idea

Volatility is **itself random** (Heston) and the price can still jump (Merton). Instantaneous variance $v_t$ mean-reverts to a long-run level $\theta$.

**Variance — continuous:**

$$dv_t = \kappa(\theta - v_t)\, dt + \xi\sqrt{v_t}\, dW_t^v$$

**Price — continuous:**

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa_J)\, dt + \sqrt{v_t}\, dW_t^S + (e^J - 1)\, dN_t$$

**Correlation:** $\text{Corr}(dW_t^S, dW_t^v)=\rho$ (usually negative: price down → vol up).

**Variance — simulation (not differential):**

$$v_{t+\Delta t} = v_t + \kappa(\theta - v_t)\Delta t + \xi\sqrt{v_t}\sqrt{\Delta t}\, Z_v$$

(with $v\leftarrow\max(v,0)$ in code so variance stays non-negative)

**Price — simulation (not differential):**

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \lambda\kappa_J - \tfrac{1}{2}v_t\big)\Delta t + \sqrt{v_t}\sqrt{\Delta t}\, Z_S + \sum_{i=1}^{N_{\Delta t}} J_i\Big)$$

$$\kappa_J = e^{\mu_J + \sigma_J^2/2} - 1$$

| Symbol | Meaning |
|--------|---------|
| $v_t$ | instantaneous **variance** (vol $= \sqrt{v_t}$) |
| $v_0$ | starting variance |
| $\theta$ | long-run variance |
| $\kappa$ | speed of mean reversion of $v_t$ toward $\theta$ |
| $\xi$ | vol-of-vol |
| $\rho$ | correlation between price and variance shocks |
| $\lambda,\mu_J,\sigma_J$ | Merton jump parameters (same idea as notebook 02) |


## 2. End-to-end workflow

| Step | What you do | Output |
|------|-------------|--------|
| **1. Collect prices** | Adjusted closes for ticker / regime | Price series |
| **2. Log returns** | $r_t=\ln(S_t/S_{t-1})$ | Return series |
| **3. Estimate Heston vol params** | From return dynamics / realized variance: $\theta,v_0,\kappa,\xi,\rho$ | Vol block |
| **4. Estimate jump params** | Same spirit as Merton: $\lambda,\mu_J,\sigma_J$ → $\kappa_J$ | Jump block |
| **5. Estimate / choose drift $\mu$** | Mean of returns (annualized), or risk-neutral drift later for pricing | $\mu$ |
| **6. Set design** | $S_0$, $T$, $\Delta t$, paths | Grid |
| **7. Simulate** | Each step update **both** $v_t$ and $S_t$ with correlated shocks + jumps | Paths of price **and** vol |
| **8. Use paths** | Option pricing, compare regimes, study vol clustering | Research outputs |

Heston parameters are harder than GBM’s two numbers — expect rough historical estimates first, refine later (options calibration).


## 3. How to calculate parameters from historical data

Practical **moment / proxy** recipes (good enough to understand the pipeline). Advanced MLE / particle filters / option calibration can come later.

### A. Build a variance proxy from returns

Daily returns $r_t$. A simple proxy for latent variance is squared returns or a short realized-variance window:

$$RV_t = r_t^2 \quad\text{or}\quad RV_t = \frac{1}{w}\sum_{i=0}^{w-1} r_{t-i}^2$$

Annualize when comparing to yearly $\theta$: daily variance $\times 252$.

### B. Long-run variance $\theta$ (once)

$$\hat\theta_{\text{daily}} = \overline{RV},\qquad \hat\theta = \hat\theta_{\text{daily}}\times 252$$

(if you work entirely in daily units inside the simulator, stay consistent — this notebook’s sliders use **annual** variance style with $\theta\approx 0.04$ meaning ~20% vol).

Rule of thumb: if long-run vol ≈ 20%, then $\theta \approx 0.20^2 = 0.04$.

### C. Starting variance $v_0$ (once per simulation start)

Use recent realized variance at the start date:

$$\hat v_0 = RV_{\text{recent}}\times 252$$

### D. Mean-reversion speed $\kappa$ (once)

From autocorrelation of the variance proxy. If $\mathrm{Corr}(RV_t, RV_{t+1})\approx \rho_1$, a rough discrete AR(1) link is $\rho_1 \approx e^{-\kappa\Delta t}$, so

$$\hat\kappa \approx -\frac{\ln(\rho_1)}{\Delta t}$$

(with $\Delta t=1/252$). Low $\kappa$ ⇒ vol stays high for a long time after a shock.

### E. Vol-of-vol $\xi$ (once)

Roughly from the variability of the variance proxy (method-of-moments on $\Delta RV$). Larger $\xi$ ⇒ wilder vol spikes. Typical playground values: $\xi \in [0.3, 1.0]$.

### F. Correlation $\rho$ (once)

Correlate returns with changes in the variance proxy:

$$\hat\rho = \mathrm{Corr}\big(r_t,\ \Delta RV_t\big)$$

Equity index: usually **negative** (leverage effect), e.g. $\rho \approx -0.6$.

### G. Jump block (once)

Same as Merton notebook 02: threshold jumps → $\hat\lambda,\hat\mu_J,\hat\sigma_J$, then $\kappa_J=\exp(\mu_J+\sigma_J^2/2)-1$.

### H. Drift $\mu$ (once)

$$\hat\mu = \bar{r}\times 252$$

### Data pointers

- `../data/equity/log_returns_by_regime.csv` — fit per regime
- Crisis: expect higher $\theta$, higher $\xi$, more negative jumps


## 4. Constant vs path-updating parameters

### Calibrated once (fixed for the whole run)

| Parameter | Role | Updates during a path? |
|-----------|------|------------------------|
| $\mu$ | drift | **No** |
| $\kappa,\theta,\xi,\rho$ | Heston vol dynamics | **No** |
| $v_0$ | initial variance | **No** (only the starting value) |
| $\lambda,\mu_J,\sigma_J,\kappa_J$ | jumps | **No** |
| $S_0,T,\Delta t$ | design | **No** |

### Evolve along each Monte Carlo path

| Quantity | Role | Updates during a path? |
|----------|------|------------------------|
| $v_t$ | variance state | **Yes** — random, mean-reverting path |
| $S_t$ | price | **Yes** — uses current $\sqrt{v_t}$ |
| $Z_S,Z_v$ | correlated shocks | **Yes** — redrawn each step |
| $N_{\Delta t}, J_i$ | jumps | **Yes** — redrawn each step |

**This is the big difference from GBM/Merton:** volatility is not a constant $\sigma$. The path is a **pair** $(S_t, v_t)$.

### One simulation step

1. Keep $(\mu,\kappa,\theta,\xi,\rho,v_0,\lambda,\mu_J,\sigma_J)$ fixed.
2. Draw correlated $Z_S,Z_v$ with correlation $\rho$.
3. Update $v_{t+\Delta t}$ (then clip at 0).
4. Draw jumps; update $S_{t+\Delta t}$ using **current** $v_t$ in the diffusion term.


## 5. Interactive playground

Watch the **price** panel and the **instantaneous vol** $\sqrt{v_t}$ panel together. Raise $\xi$ for wilder vol; make $\rho$ more negative for leverage effect; raise $\lambda$ for jumps on top of stochastic vol.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_heston_merton(
    mu, kappa, theta, xi, rho, v0,
    lam, mu_j, sigma_j,
    S0, T, n_steps, n_paths, seed=42,
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    kappa_j = np.exp(mu_j + 0.5 * sigma_j**2) - 1.0

    S = np.full(n_paths, S0, dtype=float)
    v = np.full(n_paths, v0, dtype=float)
    paths = np.empty((n_paths, n_steps + 1))
    vol_paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = S
    vol_paths[:, 0] = np.sqrt(np.maximum(v, 0.0))
    log_rets = np.empty((n_paths, n_steps))

    for i in range(n_steps):
        z1 = rng.standard_normal(n_paths)
        z2 = rng.standard_normal(n_paths)
        dWs = z1
        dWv = rho * z1 + np.sqrt(max(1.0 - rho**2, 0.0)) * z2

        v_pos = np.maximum(v, 0.0)
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * dWv
        v = np.maximum(v, 0.0)

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump = np.zeros(n_paths)
        mask = n_jumps > 0
        jump[mask] = (
            n_jumps[mask] * mu_j
            + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
        )

        incr = (mu - 0.5 * v_pos - lam * kappa_j) * dt + np.sqrt(v_pos * dt) * dWs + jump
        S = S * np.exp(incr)
        paths[:, i + 1] = S
        vol_paths[:, i + 1] = np.sqrt(v)
        log_rets[:, i] = incr

    t = np.linspace(0, T, n_steps + 1)
    return t, paths, vol_paths, log_rets

def plot_heston_merton(
    mu=0.05, kappa=2.0, theta=0.04, xi=0.5, rho=-0.6, v0=0.04,
    lam=0.3, mu_j=-0.05, sigma_j=0.10,
    S0=100.0, T=1.0, n_steps=252, n_paths=40,
):
    t, paths, vol_paths, log_rets = simulate_heston_merton(
        mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.8)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean")
    axes[0].set_title("Price paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].plot(t, vol_paths.T, alpha=0.35, lw=0.8)
    axes[1].plot(t, vol_paths.mean(axis=0), color="black", lw=2)
    axes[1].set_title("Instantaneous vol √v")
    axes[1].set_xlabel("years")

    axes[2].hist(log_rets.ravel(), bins=80, density=True, alpha=0.75, color="seagreen")
    axes[2].set_title("Step log returns")
    axes[2].set_xlabel("log return")

    fig.suptitle(
        f"κ={kappa:.1f}, θ={theta:.3f}, ξ={xi:.2f}, ρ={rho:.2f}, λ={lam:.2f}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_heston_merton,
    mu=FloatSlider(value=0.05, min=-0.10, max=0.30, step=0.01, description="μ"),
    kappa=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description="κ"),
    theta=FloatSlider(value=0.04, min=0.005, max=0.20, step=0.005, description="θ var"),
    xi=FloatSlider(value=0.50, min=0.05, max=2.0, step=0.05, description="ξ volvol"),
    rho=FloatSlider(value=-0.60, min=-0.95, max=0.95, step=0.05, description="ρ"),
    v0=FloatSlider(value=0.04, min=0.005, max=0.20, step=0.005, description="v0"),
    lam=FloatSlider(value=0.30, min=0.0, max=3.0, step=0.1, description="λ"),
    mu_j=FloatSlider(value=-0.05, min=-0.30, max=0.15, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.10, min=0.01, max=0.40, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="T"),
    n_steps=IntSlider(value=252, min=50, max=750, step=10, description="steps"),
    n_paths=IntSlider(value=40, min=5, max=120, step=5, description="paths"),
);